# RK1 Houseplant SFT — Colab T4

Стек (pinned): `transformers==4.57.1` + `trl==0.24.0` + `peft==0.17.1` + `bitsandbytes` (Linux).

Перед запуском:
1. Runtime → GPU (T4).
2. Если после старого `--force-reinstall` сломан numpy/torch: **Runtime → Disconnect and delete runtime**, затем новый сеанс.
3. Склонируйте репозиторий (ячейка ниже).
4. Не коммитьте `OPENROUTER_API_KEY`; для judge — Colab Secrets.
5. Не используйте `pip install -U` / `--force-reinstall` на весь файл требований.

In [1]:
!git clone https://github.com/Schaft-s/LLM || (cd LLM && git pull)
REPO = '/content/LLM'
import os, sys
os.chdir(REPO)
sys.path.insert(0, REPO)
!nvidia-smi || true
!ls data/processed/plant_train.jsonl

Cloning into 'LLM'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 133 (delta 50), reused 111 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 720.62 KiB | 20.02 MiB/s, done.
Resolving deltas: 100% (50/50), done.
Sun Nov  1 20:53:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+========

In [2]:
# Safe install: only our pins. NEVER --force-reinstall the whole Colab env.
# Needs bitsandbytes>=0.49.2 for Colab CUDA 13 (older 0.45 = CPU-only → QLoRA crash).
# If numpy/_slice ImportError already happened: Disconnect and delete runtime first.
!bash scripts/colab_install.sh
!python scripts/smoke_colab_stack.py
!python -c "import torch,bitsandbytes as b; from pathlib import Path; print('cuda', torch.cuda.is_available(), torch.version.cuda); print([p.name for p in Path(b.__file__).parent.glob('libbitsandbytes_cuda*.so')])"

Found existing installation: transformers 5.21.0
Uninstalling transformers-5.21.0:
  Successfully uninstalled transformers-5.21.0
Found existing installation: peft 0.21.2
Uninstalling peft-0.21.2:
  Successfully uninstalled peft-0.21.2
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
I0000 00:00:1793566439.1

## Pilot (20 steps): train → save adapter

In [3]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'scripts/train_qlora.py', '--pilot'],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)
print('Pilot OK → artifacts/weights/pilot_qlora')

{'loss': 2.9521, 'grad_norm': 0.8790335059165955, 'learning_rate': 6.227427435703997e-05, 'entropy': 2.1414162114262583, 'num_tokens': 31152.0, 'mean_token_accuracy': 0.4238843057304621, 'epoch': 0.11}
{'loss': 2.2943, 'grad_norm': 0.6978644132614136, 'learning_rate': 6.819348298638839e-07, 'entropy': 2.320702561736107, 'num_tokens': 62124.0, 'mean_token_accuracy': 0.5065607145428658, 'epoch': 0.22}
{'train_runtime': 173.7194, 'train_samples_per_second': 0.921, 'train_steps_per_second': 0.115, 'train_loss': 2.6232038497924806, 'epoch': 0.22}
Saved adapter to /content/LLM/artifacts/weights/pilot_qlora

Pilot OK → artifacts/weights/pilot_qlora


## Reload smoke + 1 generation

In [4]:
from src.train.generate import generate_chat, load_base_or_adapter
from src.train.sft_qlora import load_yaml

model_cfg = load_yaml('configs/model.yaml')
model, tok = load_base_or_adapter(model_cfg, adapter_dir='artifacts/weights/pilot_qlora')
prompt = (
    'My snake plant has mushy leaves at the base after weekly watering in low light. '
    'Give exactly 3 numbered steps I should try first.'
)
text = generate_chat(
    model, tok, prompt,
    system_prompt=model_cfg.get('system_prompt', '').strip(),
    max_new_tokens=256,
    temperature=0.2,
)
print(text)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

1. Move to brighter light
2. Water less often
3. Prune off mushy parts


## S1 full QLoRA (after pilot OK)

In [7]:
# Uncomment for full run
result = subprocess.run([sys.executable, 'scripts/train_qlora.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode: print(result.stderr); raise SystemExit(result.returncode)
print('Uncomment this cell after pilot succeeds')

{'loss': 3.0379, 'grad_norm': 1.1206125020980835, 'learning_rate': 9.890738003669029e-05, 'entropy': 2.101593238115311, 'num_tokens': 31152.0, 'mean_token_accuracy': 0.4162517078220844, 'epoch': 0.11}
{'loss': 2.2413, 'grad_norm': 0.47558221220970154, 'learning_rate': 9.24024048078213e-05, 'entropy': 2.24520151168108, 'num_tokens': 62124.0, 'mean_token_accuracy': 0.5180790349841118, 'epoch': 0.22}
{'loss': 2.1289, 'grad_norm': 0.3763921558856964, 'learning_rate': 8.07830737662829e-05, 'entropy': 2.0746614545583726, 'num_tokens': 95786.0, 'mean_token_accuracy': 0.5384239390492439, 'epoch': 0.32}
{'loss': 2.1019, 'grad_norm': 0.3432149887084961, 'learning_rate': 6.545084971874738e-05, 'entropy': 2.018201674520969, 'num_tokens': 128540.0, 'mean_token_accuracy': 0.5439549569040537, 'epoch': 0.43}
{'loss': 2.1047, 'grad_norm': 0.36866435408592224, 'learning_rate': 4.825502516487497e-05, 'entropy': 2.054229626059532, 'num_tokens': 163366.0, 'mean_token_accuracy': 0.541437916830182, 'epoch': 

## Eval helpers

После обучения сгенерируйте ответы base vs adapter одинаковыми decoding settings, затем:

```bash
python scripts/score_generations.py --mode general --predictions reports/metrics/general_dev_sft.jsonl --baseline reports/metrics/general_dev_base.jsonl
python scripts/score_generations.py --mode format --predictions reports/metrics/format_prompts_sft.jsonl
python scripts/run_judge.py --input reports/metrics/plant_dev_pair.jsonl --limit 20
```

Правило чекпоинта: plant_dev не хуже базы; general_dev drop ≤ 5 п.п.

In [14]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [15]:
!python scripts/score_generations.py --mode general --predictions reports/metrics/general_dev_sft.jsonl --baseline reports/metrics/general_dev_base.jsonl
!python scripts/score_generations.py --mode format --predictions reports/metrics/format_prompts_sft.jsonl
!python scripts/run_judge.py --input reports/metrics/plant_dev_pair.jsonl --limit 20

Traceback (most recent call last):
  File "/content/LLM/scripts/score_generations.py", line 62, in <module>
    main()
    ~~~~^^
  File "/content/LLM/scripts/score_generations.py", line 27, in main
    rows = read_jsonl(args.predictions)
  File "/content/LLM/src/data/io_jsonl.py", line 24, in read_jsonl
    with path.open(encoding="utf-8") as f:
         ~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/pathlib/_local.py", line 537, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'reports/metrics/general_dev_sft.jsonl'
Traceback (most recent call last):
  File "/content/LLM/scripts/score_generations.py", line 62, in <module>
    main()
    ~~~~^^
  File "/content/LLM/scripts/score_generations.py", line 27, in main
    rows = read_jsonl(args.predictions)
  File "/content/LLM/src/data/io_jsonl.py", line 24, in read_jsonl
    with

In [13]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rk1_houseplant
!cp -r /content/LLM/artifacts/weights/s1_plant_qlora /content/drive/MyDrive/rk1_houseplant/

Mounted at /content/drive


In [9]:
from src.eval.checkers import check_three_numbered_steps
demo_ok = '''1. Raise humidity.\n2. Trim brown tips.\n3. Check drainage.'''
demo_bad = '''1. Water less.\n2. Water less.\n3. Water less.'''
print(check_three_numbered_steps(demo_ok))
print(check_three_numbered_steps(demo_bad))

CheckResult(name='three_numbered_steps', passed=True, detail='ok')
CheckResult(name='three_numbered_steps', passed=False, detail='duplicated step content')
